In [1]:
# DMML Assignment 2
# Step 1: Data Ingestion

print("MOVIE RECOMMENDATION PIPELINE")
print("=" * 45)
print("Data Ingestion")
print("Environment: Google Colab")

MOVIE RECOMMENDATION PIPELINE
Data Ingestion
Environment: Google Colab


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"
os.chdir(PROJECT_DIR)
print("Current project directory:")
print(os.getcwd())

Current project directory:
/content/drive/MyDrive/recommendation-pipeline


In [4]:
RAW_DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw")

print("Files available in data/raw:")
print("=" * 40)

for file in os.listdir(RAW_DATA_DIR):
    print(file)

Files available in data/raw:
links.csv
tags.csv
ratings.csv
movies.csv
users.csv


In [5]:
import pandas as pd

# Load MovieLens datasets
movies_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "movies.csv"))
ratings_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "ratings.csv"))
tags_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "tags.csv"))
links_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "links.csv"))

print("All datasets loaded successfully!")
print("-" * 45)

print("Movies:", movies_df.shape)
print("Ratings:", ratings_df.shape)
print("Tags:", tags_df.shape)
print("Links:", links_df.shape)

All datasets loaded successfully!
---------------------------------------------
Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)
Links: (9742, 3)


In [6]:
print("MOVIES DATASET")
display(movies_df.head())

print("RATINGS DATASET")
display(ratings_df.head())

print("TAGS DATASET")
display(tags_df.head())

print("LINKS DATASET")
display(links_df.head())

MOVIES DATASET


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


RATINGS DATASET


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


TAGS DATASET


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


LINKS DATASET


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [7]:
print("MOVIES DATASET")
movies_df.info()

print("\nRATINGS DATASET")
ratings_df.info()

print("\nTAGS DATASET")
tags_df.info()

print("\nLINKS DATASET")
links_df.info()

MOVIES DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB

RATINGS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB

TAGS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 

In [8]:
print("MISSING VALUES REPORT")
print("=" * 45)

datasets = {
    "Movies": movies_df,
    "Ratings": ratings_df,
    "Tags": tags_df,
    "Links": links_df
}

for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.isnull().sum())

MISSING VALUES REPORT

Movies:
movieId    0
title      0
genres     0
dtype: int64

Ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Tags:
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

Links:
movieId    0
imdbId     0
tmdbId     8
dtype: int64


In [9]:
print("DUPLICATE RECORDS REPORT")
print("=" * 45)

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

DUPLICATE RECORDS REPORT
Movies: 0 duplicate rows
Ratings: 0 duplicate rows
Tags: 0 duplicate rows
Links: 0 duplicate rows


In [10]:
print("RATING VALIDATION")
print("=" * 40)

print("Minimum rating:", ratings_df["rating"].min())
print("Maximum rating:", ratings_df["rating"].max())

invalid_ratings = ratings_df[
    (ratings_df["rating"] < 0.5) |
    (ratings_df["rating"] > 5.0)
]

print("Invalid ratings:", len(invalid_ratings))

RATING VALIDATION
Minimum rating: 0.5
Maximum rating: 5.0
Invalid ratings: 0


In [11]:
print("MOVIE ID VALIDATION")
print("=" * 40)

movie_ids = set(movies_df["movieId"])

invalid_movie_ratings = ratings_df[
    ~ratings_df["movieId"].isin(movie_ids)
]

print("Ratings with invalid movie IDs:", len(invalid_movie_ratings))

MOVIE ID VALIDATION
Ratings with invalid movie IDs: 0


In [12]:
# Create Parquet copies of the datasets

PROCESSED_DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

movies_df.to_parquet(
    os.path.join(PROCESSED_DATA_DIR, "movies.parquet"),
    index=False
)

ratings_df.to_parquet(
    os.path.join(PROCESSED_DATA_DIR, "ratings.parquet"),
    index=False
)

tags_df.to_parquet(
    os.path.join(PROCESSED_DATA_DIR, "tags.parquet"),
    index=False
)

links_df.to_parquet(
    os.path.join(PROCESSED_DATA_DIR, "links.parquet"),
    index=False
)

print("All four datasets saved as Parquet files successfully!")

All four datasets saved as Parquet files successfully!


In [13]:
print("FILES IN PROCESSED DATA FOLDER")
print("=" * 45)

for file in sorted(os.listdir(PROCESSED_DATA_DIR)):
    print(file)

FILES IN PROCESSED DATA FOLDER
links.parquet
movies.parquet
ratings.parquet
tags.parquet
users.parquet


In [14]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"
RAW_DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw")

ratings_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "ratings.csv"))

print("Ratings dataset loaded successfully!")
print("Shape:", ratings_df.shape)

Ratings dataset loaded successfully!
Shape: (100836, 4)


In [15]:
unique_users = sorted(ratings_df["userId"].unique())

print("Total unique users:", len(unique_users))
print("First 10 user IDs:", unique_users[:10])

Total unique users: 610
First 10 user IDs: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [16]:
import numpy as np
import pandas as pd

# Ensuring reproducible synthetic data
np.random.seed(42)

occupations = [
    "Student", "Engineer", "Teacher", "Doctor",
    "Artist", "Business", "Researcher", "Other"
]

locations = [
    "Toronto", "Vancouver", "Calgary",
    "Montreal", "Ottawa", "Edmonton"
]

# Creating synthetic user records
users_df = pd.DataFrame({
    "userId": unique_users,
    "age": np.random.randint(18, 65, size=len(unique_users)),
    "occupation": np.random.choice(occupations, size=len(unique_users)),
    "location": np.random.choice(locations, size=len(unique_users))
})

print("Synthetic user dataset created successfully!")
print("Total synthetic users:", len(users_df))

display(users_df.head(10))

Synthetic user dataset created successfully!
Total synthetic users: 610


,userId,age,occupation,location
0,1,56,Student,Toronto
1,2,46,Engineer,Toronto
2,3,32,Teacher,Toronto
3,4,60,Engineer,Toronto
4,5,25,Engineer,Toronto
5,6,38,Artist,Vancouver
6,7,56,Artist,Edmonton
7,8,36,Artist,Ottawa
8,9,40,Business,Calgary
9,10,28,Teacher,Calgary


In [17]:
# synthetic user data in CSV - saving
users_csv_path = os.path.join(RAW_DATA_DIR, "users.csv")
users_df.to_csv(users_csv_path, index=False)

# synthetic user data in Parquet - saving
users_parquet_path = os.path.join(PROJECT_DIR, "data", "processed", "users.parquet")
users_df.to_parquet(users_parquet_path, index=False)

print("User dataset saved successfully!")
print("CSV location:", users_csv_path)
print("Parquet location:", users_parquet_path)

User dataset saved successfully!
CSV location: /content/drive/MyDrive/recommendation-pipeline/data/raw/users.csv
Parquet location: /content/drive/MyDrive/recommendation-pipeline/data/processed/users.parquet


In [18]:
print("Checking saved user files...")

print("users.csv exists:",
      os.path.exists(os.path.join(RAW_DATA_DIR, "users.csv")))

print("users.parquet exists:",
      os.path.exists(os.path.join(PROJECT_DIR, "data", "processed", "users.parquet")))

Checking saved user files...
users.csv exists: True
users.parquet exists: True


In [19]:
users_check = pd.read_csv(
    os.path.join(RAW_DATA_DIR, "users.csv")
)

print("CSV rows and columns:", users_check.shape)
print("Unique user IDs:", users_check["userId"].nunique())

display(users_check.head())

CSV rows and columns: (610, 4)
Unique user IDs: 610


,userId,age,occupation,location
0,1,56,Student,Toronto
1,2,46,Engineer,Toronto
2,3,32,Teacher,Toronto
3,4,60,Engineer,Toronto
4,5,25,Engineer,Toronto


In [20]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"
RAW_DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw")

movies_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "movies.csv"))
tags_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "tags.csv"))
links_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "links.csv"))

print("DATA INGESTION SUMMARY")
print("=" * 40)

print("Movies:", movies_df.shape)
print("Ratings:", ratings_df.shape)
print("Tags:", tags_df.shape)
print("Links:", links_df.shape)
print("Synthetic Users:", users_df.shape)

print("\nAll five datasets are available for the pipeline.")

DATA INGESTION SUMMARY
Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)
Links: (9742, 3)
Synthetic Users: (610, 4)

All five datasets are available for the pipeline.


In [21]:
print("RAW DATA FILES:")
print(os.listdir(RAW_DATA_DIR))

print("\nPROCESSED DATA FILES:")
print(os.listdir(os.path.join(PROJECT_DIR, "data", "processed")))

RAW DATA FILES:
['links.csv', 'tags.csv', 'ratings.csv', 'movies.csv', 'users.csv']

PROCESSED DATA FILES:
['tags.parquet', 'ratings.parquet', 'movies.parquet', 'users.parquet', 'links.parquet']


# Phase 2: Data Validation

This phase validates all five datasets using the
Pandera schemas created in the Setup notebook.

Validation checks include:
- Data types
- Missing values
- Duplicate records
- Rating ranges
- Unique user and movie IDs

In [22]:
import sys
import os

VALIDATION_DIR = os.path.join(
    PROJECT_DIR,
    "src",
    "validation"
)

if VALIDATION_DIR not in sys.path:
    sys.path.insert(0, VALIDATION_DIR)

print("Validation module path configured successfully!")

Validation module path configured successfully!


In [23]:
%pip install -q pandera

In [24]:
import pandera as pa

print("Pandera version:", pa.__version__)
print("Pandera installed successfully!")

Pandera version: 0.36.0
Pandera installed successfully!


In [25]:
from schema import (
    ratings_schema,
    users_schema,
    movies_schema,
    tags_schema,
    links_schema
)

from validate_data import validate_dataset

print("All validation modules imported successfully!")

All validation modules imported successfully!


In [26]:
import importlib
import schema
import validate_data

importlib.reload(schema)
importlib.reload(validate_data)

from schema import (
    ratings_schema,
    users_schema,
    movies_schema,
    tags_schema,
    links_schema
)

from validate_data import validate_dataset

print("All validation modules imported successfully!")

All validation modules imported successfully!


In [27]:
validated_ratings = validate_dataset(
    ratings_df,
    ratings_schema,
    "Ratings"
)

Ratings: VALIDATION PASSED


In [28]:
#Validate the users dataset
validated_users = validate_dataset(
    users_df,
    users_schema,
    "Users"
)

# Validate the movies dataset
validated_movies = validate_dataset(
    movies_df,
    movies_schema,
    "Movies"
)

# Validate the tags datasets
validated_tags = validate_dataset(
    tags_df,
    tags_schema,
    "Tags"
)

# Validate the links datasets

validated_links = validate_dataset(
    links_df,
    links_schema,
    "Links"
)

Users: VALIDATION PASSED
Movies: VALIDATION PASSED
Tags: VALIDATION PASSED
Links: VALIDATION PASSED


In [29]:
validation_results = {
    "Ratings": validated_ratings,
    "Users": validated_users,
    "Movies": validated_movies,
    "Tags": validated_tags,
    "Links": validated_links
}

validation_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Status": "PASSED" if result is not None else "FAILED",
        "Rows": len(result) if result is not None else 0
    }
    for name, result in validation_results.items()
])

display(validation_summary)

,Dataset,Status,Rows
0,Ratings,PASSED,100836
1,Users,PASSED,610
2,Movies,PASSED,9742
3,Tags,PASSED,3683
4,Links,PASSED,9742


In [30]:
total_datasets = len(validation_summary)
passed_datasets = (
    validation_summary["Status"] == "PASSED"
).sum()

failed_datasets = total_datasets - passed_datasets

print("DATA VALIDATION SUMMARY")
print("=" * 40)
print("Total datasets:", total_datasets)
print("Passed:", passed_datasets)
print("Failed:", failed_datasets)

if failed_datasets == 0:
    print("\nOverall result: ALL VALIDATIONS PASSED")
else:
    print("\nOverall result: SOME VALIDATIONS FAILED")

DATA VALIDATION SUMMARY
Total datasets: 5
Passed: 5
Failed: 0

Overall result: ALL VALIDATIONS PASSED


In [31]:
REPORT_DIR = os.path.join(PROJECT_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

report_path = os.path.join(
    REPORT_DIR,
    "validation_report.csv"
)

validation_summary.to_csv(report_path, index=False)

print("Validation report saved:", report_path)

Validation report saved: /content/drive/MyDrive/recommendation-pipeline/reports/validation_report.csv


In [32]:
# Verifying the saved report
report_check = pd.read_csv(
    os.path.join(PROJECT_DIR, "reports", "validation_report.csv")
)

display(report_check)

,Dataset,Status,Rows
0,Ratings,PASSED,100836
1,Users,PASSED,610
2,Movies,PASSED,9742
3,Tags,PASSED,3683
4,Links,PASSED,9742


In [33]:
# ==========================================
# DUPLICATE AND REFERENTIAL INTEGRITY CHECKS
# ==========================================

# 1. Define the datasets dictionary

datasets = {
    "Movies": movies_df,
    "Ratings": ratings_df,
    "Tags": tags_df,
    "Links": links_df,
    "Users": users_df
}

# 2. Check duplicate records

print("DUPLICATE RECORD CHECK")
print("=" * 45)

duplicate_results = {}

for name, dataframe in datasets.items():
    duplicates = dataframe.duplicated().sum()
    duplicate_results[name] = duplicates

    print(f"{name}: {duplicates} duplicate rows")


# 3. Check referential integrity

print("\nREFERENTIAL INTEGRITY CHECK")
print("=" * 45)

invalid_rating_users = ~ratings_df["userId"].isin(
    users_df["userId"]
)

invalid_rating_movies = ~ratings_df["movieId"].isin(
    movies_df["movieId"]
)

invalid_tag_users = ~tags_df["userId"].isin(
    users_df["userId"]
)

invalid_tag_movies = ~tags_df["movieId"].isin(
    movies_df["movieId"]
)

print("Ratings with unknown users:",
      invalid_rating_users.sum())

print("Ratings with unknown movies:",
      invalid_rating_movies.sum())

print("Tags with unknown users:",
      invalid_tag_users.sum())

print("Tags with unknown movies:",
      invalid_tag_movies.sum())


# 4. Overall results

print("\nOVERALL CHECK RESULTS")
print("=" * 45)

print("Total duplicate rows:",
      sum(duplicate_results.values()))

total_invalid_references = (
    invalid_rating_users.sum()
    + invalid_rating_movies.sum()
    + invalid_tag_users.sum()
    + invalid_tag_movies.sum()
)

print("Total invalid references:",
      total_invalid_references)

DUPLICATE RECORD CHECK
Movies: 0 duplicate rows
Ratings: 0 duplicate rows
Tags: 0 duplicate rows
Links: 0 duplicate rows
Users: 0 duplicate rows

REFERENTIAL INTEGRITY CHECK
Ratings with unknown users: 0
Ratings with unknown movies: 0
Tags with unknown users: 0
Tags with unknown movies: 0

OVERALL CHECK RESULTS
Total duplicate rows: 0
Total invalid references: 0


In [34]:
integrity_report = pd.DataFrame({
    "Check": [
        "Duplicate records",
        "Ratings with unknown users",
        "Ratings with unknown movies",
        "Tags with unknown users",
        "Tags with unknown movies"
    ],
    "Invalid_Count": [
        sum(duplicate_results.values()),
        invalid_rating_users.sum(),
        invalid_rating_movies.sum(),
        invalid_tag_users.sum(),
        invalid_tag_movies.sum()
    ]
})

integrity_report.to_csv(
    os.path.join(PROJECT_DIR, "reports", "integrity_report.csv"),
    index=False
)

display(integrity_report)

print("Integrity report saved successfully!")

,Check,Invalid_Count
0,Duplicate records,0
1,Ratings with unknown users,0
2,Ratings with unknown movies,0
3,Tags with unknown users,0
4,Tags with unknown movies,0


Integrity report saved successfully!


In [35]:
schema_report = validation_summary.copy()

integrity_results = [
    {
        "Dataset": row["Check"],
        "Status": "PASSED" if row["Invalid_Count"] == 0 else "FAILED",
        "Rows": row["Invalid_Count"]
    }
    for _, row in integrity_report.iterrows()
]

integrity_summary = pd.DataFrame(integrity_results)

final_validation_report = pd.concat(
    [schema_report, integrity_summary],
    ignore_index=True
)

final_report_path = os.path.join(
    PROJECT_DIR,
    "reports",
    "final_validation_report.csv"
)

final_validation_report.to_csv(final_report_path, index=False)

display(final_validation_report)

print("Final validation report saved successfully!")

,Dataset,Status,Rows
0,Ratings,PASSED,100836
1,Users,PASSED,610
2,Movies,PASSED,9742
3,Tags,PASSED,3683
4,Links,PASSED,9742
5,Duplicate records,PASSED,0
6,Ratings with unknown users,PASSED,0
7,Ratings with unknown movies,PASSED,0
8,Tags with unknown users,PASSED,0
9,Tags with unknown movies,PASSED,0


Final validation report saved successfully!


In [36]:
final_check = pd.read_csv(final_report_path)

print("Total validation checks:", len(final_check))
print("Passed checks:", (final_check["Status"] == "PASSED").sum())
print("Failed checks:", (final_check["Status"] == "FAILED").sum())

Total validation checks: 10
Passed checks: 10
Failed checks: 0
